In [1]:
#install required libs
!pip install transformers datasets torch


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Restart he Kernel

### Intro

##### Large Language Models such as Falcon, LLaMA, etc. are pretrained transformer models initially trained to predict the next token given some input text. They typically have billions of parameters and have been trained on trillions of tokens for an extended period of time. As a result, these models become quite powerful and versatile, and you can use them to solve multiple NLP tasks out of the box by instructing the models with natural language prompts.
##### Designing such prompts to ensure the optimal output is often called “prompt engineering”. Prompt engineering is an iterative process that requires a fair amount of experimentation. Natural languages are much more flexible and expressive than programming languages, however, they can also introduce some ambiguity. At the same time, prompts in natural language are quite sensitive to changes. Even minor modifications in prompts can lead to wildly different outputs.

##### The majority of modern LLMs are decoder-only transformers. Some examples include: LLaMA, Llama2, Falcon, GPT2. However, you may encounter encoder-decoder transformer LLMs as well, for instance, Flan-T5 and BART.
##### Encoder-decoder-style models are typically used in generative tasks where the output heavily relies on the input, for example, in translation and summarization. The decoder-only models are used for all other types of generative tasks.

### Base vs instruct/chat models

##### Base models are excellent at completing the text when given an initial prompt, however, they are not ideal for NLP tasks where they need to follow instructions, or for conversational use. This is where the instruct (chat) versions come in. These checkpoints are the result of further fine-tuning of the pre-trained base versions on instructions and conversational data. This additional fine-tuning makes them a better choice for many NLP tasks.

#####  Let’s illustrate some simple prompts that you can use with falcon-7b-instruct to solve some common NLP tasks.

In [1]:
from transformers import pipeline, AutoTokenizer
import torch

torch.manual_seed(0)
model = "tiiuae/falcon-7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


### Text classification

##### One of the most common forms of text classification is sentiment analysis, which assigns a label like “positive”, “negative”, or “neutral” to a sequence of text. Let’s write a prompt that instructs the model to classify a given text (a movie review). We’ll start by giving the instruction, and then specifying the text to classify.

In [2]:
torch.manual_seed(0)
prompt = """Classify the text into neutral, negative or positive. 
Text: This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.
Sentiment:
"""

sequences = pipe(
    prompt,
    max_new_tokens=10,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=

Result: Classify the text into neutral, negative or positive. 
Text: This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.
Sentiment:
Positive


### Named Entity Recognition

##### Named Entity Recognition (NER) is a task of finding named entities in a piece of text, such as a person, location, or organization. Let’s modify the instructions in the prompt to make the LLM perform this task.

In [3]:
torch.manual_seed(1)
prompt = """Return a list of named entities in the text.
Text: The Golden State Warriors are an American professional basketball team based in San Francisco.
Named entities:
"""

sequences = pipe(
    prompt,
    max_new_tokens=15,
    return_full_text = False,    
)

for seq in sequences:
    print(f"{seq['generated_text']}")

[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- Golden State Warriors
- San Francisco


### Translation

##### Another task LLMs can perform is translation. You can choose to use encoder-decoder models for this task, however, here, for the simplicity of the examples, we’ll keep using Falcon-7b-instruct, which does a decent job.

In [4]:
torch.manual_seed(2)
prompt = """Translate the English text to Italian.
Text: Sometimes, I've believed as many as six impossible things before breakfast.
Translation:
"""

sequences = pipe(
    prompt,
    max_new_tokens=20,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)

for seq in sequences:
    print(f"{seq['generated_text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A volte, ho creduto in sei impossibili cose prima colazione.


### Text summarization

##### Similar to the translation, text summarization is another generative task where the output heavily relies on the input, and encoder-decoder models can be a better choice. However, decoder-style models can be used for this task as well. Previously, we have placed the instructions at the very beginning of the prompt. However, the very end of the prompt can also be a suitable location for instructions. Typically, it’s better to place the instruction on one of the extreme ends.

In [5]:
torch.manual_seed(3)
prompt = """Permaculture is a design process mimicking the diversity, functionality and resilience of natural ecosystems. The principles and practices are drawn from traditional ecological knowledge of indigenous cultures combined with modern scientific understanding and technological innovations. Permaculture design provides a framework helping individuals and communities develop innovative, creative and effective strategies for meeting basic needs while preparing for and mitigating the projected impacts of climate change.
Write a summary of the above text.
Summary:
"""

sequences = pipe(
    prompt,
    max_new_tokens=30,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)

for seq in sequences:
    print(f"{seq['generated_text']}")

[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Permaculture is an approach mimicking natural ecosystems' diversity, functionality, and resilience. It draws on traditional ecological knowledge, scientific understanding, and technological innovations


### Question answering

##### For question answering task we can structure the prompt into the following logical components: instructions, context, question, and the leading word or phrase ("Answer:") to nudge the model to start generating the answer

In [9]:
torch.manual_seed(4)
prompt = """Answer the question using the context below.
Context: Gazpacho is a cold soup and drink made of raw, blended vegetables. Most gazpacho includes stale bread, tomato, cucumbers, onion, bell peppers, garlic, olive oil, wine vinegar, water, and salt. Northern recipes often include cumin and/or pimentón (smoked sweet paprika). Traditionally, gazpacho was made by pounding the vegetables in a mortar with a pestle; this more laborious method is still sometimes used as it helps keep the gazpacho cool and avoids the foam and silky consistency of smoothie versions made in blenders or food processors.
Question: What ingredients a common gazpacho includes?
Answer:
"""

sequences = pipe(
    prompt,
    max_new_tokens=50, #increased from 10
    do_sample= False, #True, # use greedy decoding for factual tasks
   # top_k=10,
    return_full_text = False,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: Gazpacho typically includes raw, blended vegetables such as tomatoes, cucumbers, bell peppers, onions, garlic, olive oil, wine vinegar, water, salt, and cumin or pimentón.


In [8]:
#Result 1 : In general, Gazpacho includes raw, blended - not full answer
#max_new_tokens=10 — Very limited, cuts off the answer mid-sentence. "raw, blended

### Reasoning

##### Reasoning is one of the most difficult tasks for LLMs, and achieving good results often requires applying advanced prompting techniques, like Chain-of-thought.

In [12]:
torch.manual_seed(5)
prompt = """I baked 15 muffins. I ate 2 muffins and gave 5 muffins to a neighbor. 
My partner then bought 6 more muffins and ate 2. How many muffins do we now have? """

sequences = pipe(
    prompt,
    max_new_tokens=100, #increased from 10
    do_sample=False,# use greedy decoding
    #top_k=10,
    return_full_text = False,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: 
We now have 15 + 2 + 5 + 6 = 22 muffins.


In [13]:
#Result: We have 18 muffins now. - incorrect
## falcon-7b-instruct still struggle with reasoning task

##### The answers of the model are not determenistic. In most of the cases model can fail with the correct answer. In this case, this can be due to the prompt being too basic, or due to the choice of model, after all we’ve picked the smallest version of Falcon. Reasoning is difficult for models of all sizes, but larger models are likely to perform better.

#### You can try using DeepSeek-R1 reasoning model on EPAM DIAL and compare the results with the same prompt.

<think>
Okay, let's see. I need to figure out how many muffins we have now after all these actions. Let me start from the beginning.

First, the user baked 15 muffins. So initially, there are 15 muffins. Then they ate 2 muffins. Hmm, so that should subtract 2 from the total. Let me write that down:

15 muffins - 2 muffins = 13 muffins left after eating.

Next, they gave 5 muffins to a neighbor. So we need to subtract another 5 from the remaining 13. Let me calculate that:

13 muffins - 5 muffins = 8 muffins left after giving to the neighbor.

Now, the partner comes into the picture. The partner bought 6 more muffins. That means we need to add those 6 muffins to the current total. So:

8 muffins + 6 muffins = 14 muffins after the partner buys more.

Then, the partner ate 2 muffins. So subtract 2 from the 14:

14 muffins - 2 muffins = 12 muffins left.

Wait, let me check that again to make sure I didn't make a mistake. Starting with 15, minus 2 is 13. Then minus 5 is 8. Then plus 6 is 14. Then minus 2 is 12. Yeah, that seems right. So the final number of muffins should be 12.

But let me think again step by step to confirm. Maybe I missed something. So:

1. Baked 15: total = 15.

2. Ate 2: 15 - 2 = 13.

3. Gave 5 to neighbor: 13 - 5 = 8.

4. Partner bought 6: 8 + 6 = 14.

5. Partner ate 2: 14 - 2 = 12.

Yes, that's consistent. So the answer should be 12 muffins remaining. I don't see any errors in the calculations. Each step is straightforward addition or subtraction. So I think that's correct.
</think>

Let's break down the problem step by step:

1. **Initial muffins baked**: 15
2. **After you ate 2 muffins**:
   \( 15 - 2 = 13 \) muffins left.
3. **After giving 5 to a neighbor**:
   \( 13 - 5 = 8 \) muffins left.
4. **Partner bought 6 more muffins**:
   \( 8 + 6 = 14 \) muffins.
5. **Partner ate 2 muffins**:
   \( 14 - 2 = 12 \) muffins.

**Final Answer**: \(\boxed{12}\) muffins.

## Advanced prompting techniques

### Few-shot prompting

In [14]:
##### The basic prompts in the sections above are the examples of “zero-shot” prompts, meaning, the model has been given instructions and context, but no examples with solutions. LLMs that have been fine-tuned on instruction datasets, generally perform well on such “zero-shot” tasks. However, you may find that your task has more complexity or nuance, and, perhaps, you have some requirements for the output that the model doesn’t catch on just from the instructions. In this case, you can try the technique called few-shot prompting.

##### In few-shot prompting, we provide examples in the prompt giving the model more context to improve the performance. The examples condition the model to generate the output following the patterns in the examples.

torch.manual_seed(6)
prompt = """Text: The first human went into space and orbited the Earth on April 12, 1961.
Date: 04/12/1961
Text: The first-ever televised presidential debate in the United States took place on September 28, 1960, between presidential candidates John F. Kennedy and Richard Nixon. 
Date:"""

sequences = pipe(
    prompt,
    max_new_tokens=8,
    do_sample=True,
    top_k=10,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: Text: The first human went into space and orbited the Earth on April 12, 1961.
Date: 04/12/1961
Text: The first-ever televised presidential debate in the United States took place on September 28, 1960, between presidential candidates John F. Kennedy and Richard Nixon. 
Date: 09/28/1960



### Limitations of the few-shot prompting technique:

- While LLMs can pick up on the patterns in the examples, these technique doesn’t work well on complex reasoning tasks
- Few-shot prompting requires creating lengthy prompts. Prompts with large number of tokens can increase computation and latency. There’s also a limit to the length of the prompts.
- Sometimes when given a number of examples, models can learn patterns that you didn’t intend them to learn, e.g. that the third movie review is always negative.

### Chain-of-thought

##### Chain-of-thought (CoT) prompting is a technique that nudges a model to produce intermediate reasoning steps thus improving the results on complex reasoning tasks.

##### There are two ways of steering a model to producing the reasoning steps:

##### - few-shot prompting by illustrating examples with detailed answers to questions, showing the model how to work through a problem.
##### - by instructing the model to reason by adding phrases like “Let’s think step by step” or “Take a deep breath and work through the problem step by step.”

#### If we apply the CoT technique to the muffins example from the reasoning section and use a larger model, such as GPT3.5 or GPT4 which you can play with in the https://chat.lab.epam.com, we’ll get a significant improvement on the reasoning result:

##### Let's go through this step-by-step:
##### 1. You start with 15 muffins.
##### 2. You eat 2 muffins, leaving you with 13 muffins.
##### 3. You give 5 muffins to your neighbor, leaving you with 8 muffins.
##### 4. Your partner buys 6 more muffins, bringing the total number of muffins to 14.
##### 5. Your partner eats 2 muffins, leaving you with 12 muffins.
##### Therefore, you now have 12 muffins.

### Best practices of LLM prompting

##### The list of best practices that tend to improve the prompt results:

- When choosing the model to work with, the latest and most capable models are likely to perform better.
- Start with a simple and short prompt, and iterate from there.
- Put the instructions at the beginning of the prompt, or at the very end. When working with large context, models apply various optimizations to prevent Attention complexity from scaling quadratically. This may make a model more attentive to the beginning or end of a prompt than the middle.
- Clearly separate instructions from the text they apply to - more on this in the next section.
- Be specific and descriptive about the task and the desired outcome - its format, length, style, language, etc.
- Avoid ambiguous descriptions and instructions.
- Favor instructions that say “what to do” instead of those that say “what not to do”.
- “Lead” the output in the right direction by writing the first word (or even begin the first sentence for the model).
- Use advanced techniques like Few-shot prompting and Chain-of-thought
- Test your prompts with different models to assess their robustness.
- Version and track the performance of your prompts.